# Data Cleaning

In [1]:
from pyspark.sql import SparkSession


spark = SparkSession.builder \
    .config("spark.driver.memory", "5g") \
    .config("spark.executor.memory", "5g") \
    .appName("Open food facts") \
    .getOrCreate()

# Load the data
data = spark.read.csv(
    "/Users/mac/Desktop/Spark-Recommendation-System/data/france_df.csv",
    header=True,
)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/05/27 19:52:13 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
data.show(5)

25/05/27 19:52:26 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+------------+--------------------+-------+----------+--------------------+---------------+----------------------+----------------+--------------+---------------------+--------------------+------------------------+------------+------------------+---------+--------------+------------+--------------+--------------+-----------------+--------------+--------------------+--------------------+-------------------+-------+------------+----------+--------------------+-------------------------+--------------------+--------------------+--------------------+---------+--------------+------------------------+------+-----------+---------------+------+--------------------+--------------------+--------------------+--------------------+--------------------+-------------------------+---------+------------+------+-----------+---------+------------+----------------+-----------------+-----------+---------+--------------------+--------------------+----------------+----------------+----------+-------------+---

## Delete columns

### Date and Time related columns

In [3]:
date_time_columns = [col for col in data.columns if col.endswith("_t") or col.endswith("_datetime")]

print("Columns to be removed:", date_time_columns)

cleaned_df = data.drop(*date_time_columns)

Columns to be removed: ['created_t', 'created_datetime', 'last_modified_t', 'last_modified_datetime', 'last_updated_t', 'last_updated_datetime', 'last_image_t', 'last_image_datetime']


### Columns with just null values

In [4]:
from pyspark.sql.functions import col, count, when

null_columns = [
    c for c in cleaned_df.columns
    if cleaned_df.select(count(when(col(c).isNotNull(), c)).alias(c)).first()[0] == 0
]

print("Columns with all null values:", null_columns)


25/05/27 19:52:32 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


Columns with all null values: ['cities', 'allergens_en', 'additives', 'nutrition-score-uk_100g', 'carbohydrates-total_100g']


In [ ]:
# Columns with all null values: ['cities', 'allergens_en', 'additives', 'nutrition-score-uk_100g', 'carbohydrates-total_100g']

cleaned_df = cleaned_df.drop(*null_columns)

### products metadata

In [6]:
cleaned_df.show(5, truncate=False)

+------------+----------------------------------------------------------------------------------------------------------+-------+----------------+----------------------------------------+------------------------+------------+------------------+---------+--------------+------------+--------------+--------------+-----------------+--------------+------------------------+----------------------+-------------------+-------+------------+----------+--------------------+-------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------

In [7]:
columns_to_drop = [col for col in cleaned_df.columns if 
                   "origins" in col or 
                   "manufacturing_places" in col or
                   "cities" in col or
                   "countries" in col or
                   "owner" in col or
                   "packaging" in col or 
                   "emb_codes" in col or 
                   "countries" in col or 
                   "states" in col or 
                   "100g" in col]

cleaned_df = cleaned_df.drop(*columns_to_drop)

In [8]:
len(cleaned_df.columns)

56

In [9]:
cleaned_df.show(5, truncate=False)

+------------+----------------------------------------------------------------------------------------------------------+-------+----------------+----------------------------------------+------------------------+------------+------------------+--------------+-----------------+--------------+------------------------+----------------------+-------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------+---------------+------+--------------------------------------------------------------------------------------------------

### Non usefull columns

In [10]:
non_usefull_columns = [ "creator", "last_modified_by", "abbreviated_product_name", "categories",
                       "labels", "labels_tags", "purchase_places", "stores", "labels", "labels_en", "ingredients_text", "allergens",
                        "traces", "traces_en", "serving_size", "serving_quantity", "no_nutrition_data", "additives_n", "additives_en",
                        "nutriscore_score", "pnns_groups_1", "pnns_groups_2", "food_groups", "food_groups_en", "environmental_score_score",
                        "environmental_score_grade", "nutrient_levels_tags", "data_quality_errors_tags", "unique_scans_n", "main_category_en",
                        "image_small_url","image_ingredients_url", "image_ingredients_small_url", "image_nutrition_url", "image_nutrition_small_url",
                        "popularity_tags", "brands_tags", "brands_en", "traces_tags","food_groups_tags","completeness"]

In [11]:
cleaned_df = cleaned_df.drop(*non_usefull_columns)

len(cleaned_df.columns)

16

### Calculate Completeness of rows

#### calculate completeness of rows based on selected columns

- we will calculate the completeness of columns based on our custom columns 
- we will exclude nutrition columns because they will be used just for the front and not in our recommendation systeme

In [12]:
nutrition_cols = [col for col in cleaned_df.columns if "100g" in col]

non_nutrition_cols = [col for col in cleaned_df.columns if col not in nutrition_cols]

In [13]:
number_of_complte_columns = len(non_nutrition_cols)
print("Number of columns without nutrition columns", number_of_complte_columns)
non_nutrition_cols

Number of columns without nutrition columns 16


['code',
 'url',
 'product_name',
 'generic_name',
 'quantity',
 'brands',
 'categories_tags',
 'categories_en',
 'ingredients_tags',
 'ingredients_analysis_tags',
 'additives_tags',
 'nutriscore_grade',
 'nova_group',
 'product_quantity',
 'main_category',
 'image_url']

In [14]:
columns_to_calculate_completeness = ['product_name', 'generic_name', 'categories_en', 
                                     'ingredients_tags', 'ingredients_analysis_tags', 'main_category']

In [15]:
from pyspark.sql.functions import col, when, lit, round
from functools import reduce
import operator

non_null_exprs = [
    when((col(c).isNotNull()) | (col(c) != ""), 1).otherwise(0) for c in columns_to_calculate_completeness
]

non_null_sum = reduce(operator.add, non_null_exprs)


df_with_completeness = cleaned_df.withColumn(
    "custom_completeness",
    round(non_null_sum / lit(len(columns_to_calculate_completeness)), 3)
)

In [16]:
df_with_completeness.select("custom_completeness").describe().show()

+-------+-------------------+
|summary|custom_completeness|
+-------+-------------------+
|  count|            1148555|
|   mean| 0.4431992294665675|
| stddev|0.30928631187462785|
|    min|                0.0|
|    max|                1.0|
+-------+-------------------+



#### select rows with at least 50% completeness

In [17]:
df_filetered = df_with_completeness.filter(col("custom_completeness") > 0.5)
df_filetered.show(5, truncate=False)

+--------+--------------------------------------------------------------------------------+--------------------+------------+----------------------+-----------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [18]:
df_filetered.count()

314819

In [19]:
df_filetered.show(30, truncate=False)

+--------+-----------------------------------------------------------------------------------------------------------+--------------------------------------+-----------------------------------------------------------------------------------------------------------+----------------------------+----------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

### Delete rows where main columns are null 

In [20]:
final_cleaned_df = df_filetered.na.drop(subset=["product_name", "categories_tags", "main_category"])


In [21]:
final_cleaned_df.count()

306363

### Delete columns where main_category and categories_tags aren't in english

In [28]:
from pyspark.sql.functions import col, split, trim, explode

split_main = final_cleaned_df \
    .withColumn("main_split", split(col("main_category"), ",")) \
    .withColumn("main_exploded", explode("main_split")) \
    .withColumn("main_exploded", trim(col("main_exploded")))

split_categories = final_cleaned_df \
    .withColumn("cat_split", split(col("categories_tags"), ",")) \
    .withColumn("cat_exploded", explode("cat_split")) \
    .withColumn("cat_exploded", trim(col("cat_exploded")))

non_en_main_codes = split_main.filter(~col("main_exploded").startswith("en:")).select("code").distinct()
non_en_cat_codes = split_categories.filter(~col("cat_exploded").startswith("en:")).select("code").distinct()

non_en_codes = non_en_main_codes.union(non_en_cat_codes).distinct()

english_only_df = final_cleaned_df.join(non_en_codes, on="code", how="left_anti")


- check if the non english words where deleted

In [23]:
from pyspark.sql.functions import col, explode, split, trim

non_en_categories = english_only_df.select(
    explode(split(col("categories_tags"), ",")).alias("tag")
).withColumn("tag", trim(col("tag"))) \
 .filter(~col("tag").startswith("en:"))

non_en_categories.count()


0

- count number of the final dataframe 

In [24]:
english_only_df.count()

252781

In [29]:
english_only_df.show(5, truncate=False)

+-------------+---------------------------------------------------------------------------------------------+------------------------+------------+---------+-----------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [30]:
english_only_df.columns

['code',
 'url',
 'product_name',
 'generic_name',
 'quantity',
 'brands',
 'categories_tags',
 'categories_en',
 'ingredients_tags',
 'ingredients_analysis_tags',
 'additives_tags',
 'nutriscore_grade',
 'nova_group',
 'product_quantity',
 'main_category',
 'image_url',
 'custom_completeness']

## Save the cleaned dataset

In [32]:
english_only_df.write.option("header", True).mode("overwrite").csv("/Users/mac/Desktop/Spark-Recommendation-System/data/final_data_cleaned.csv")

In [33]:
load = spark.read.csv("/Users/mac/Desktop/Spark-Recommendation-System/data/final_data_cleaned.csv", header=True)
load.show(5, truncate=False)

+-------------+---------------------------------------------------------------------------------------------------------------------------+--------------------------------------------------------------+-----------------------------------------------------------------------+----------+-----------------------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------